# Convolutional Neural Network

### Importing the libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    horizontal_flip = True)

In [ ]:
training_set = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical')

## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[223, 224, 3]))

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a third convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=126, activation='relu'))

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=3, activation='softmax'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

## Part 4 - Making prediction

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image

folder = "dataset/validation"        # folder with the images to predict
img_size = (224, 224)                       # must match training
batch_size = 32
class_names = list(training_set.class_indices.keys())

# collect all image files in the folder
files = sorted(
    f for ext in ("*.png", "*.jpg", "*.jpeg")
    for f in glob.glob(os.path.join(folder, ext))
)
print(f"Found {len(files)} images")

rows = []
for start in range(0, len(files), batch_size):
    batch_files = files[start:start + batch_size]

    batch = np.stack([
        image.img_to_array(image.load_img(f, target_size=img_size))
        for f in batch_files
    ])
    # batch = batch / 255.0   # see the note below

    probs = cnn.predict(batch, verbose=0)    # shape (n, num_classes)

    for f, p in zip(batch_files, probs):
        i = np.argmax(p)
        rows.append({
            "file": os.path.basename(f),
            "prediction": class_names[i],
            "confidence": round(float(p[i]) * 100, 1),
            **{name: round(float(v), 3) for name, v in zip(class_names, p)},
        })

df = pd.DataFrame(rows)
print(df)
df.to_csv("predictions.csv", index=False)